[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ox-vgg/wise/blob/main/notebooks/image_search.ipynb)

# Demo of Image Search using WISE

Welcome! This notebook introduces **[WISE](https://www.robots.ox.ac.uk/~vgg/software/wise/)**, an open-source search engine that lets you search image collections using natural language and example images as search query.

In the next few cells you will:

1. Install WISE
2. Choose a small demo dataset
3. Extract image features using a vision-language model
4. Build a search index
5. Search the collection interactively in your browser

No programming experience is required — just run each cell in order by clicking the run button on its left (or pressing `Shift`+`Enter`).

> For the best experience, we recommend using the Google Chrome browser and signing in to your Google account. Some features of this notebook may not work in other browsers or without being signed in.
>
> The first run takes a few minutes. Installing WISE and downloading the vision language model (`ViT-B-16-SigLIP2-512`, a few hundred MB) happens only once per Colab session.
>
> A GPU makes this much faster. This notebook requests a GPU automatically. If you ever need to set it manually, go to **Runtime > Change runtime type > Hardware accelerator > GPU**.

---

Created by Abhishek Dutta (adutta@robots.ox.ac.uk), Visual Geometry Group, University of Oxford.


## Step 1 — Install WISE

This downloads the WISE software, installs its Python dependencies, and builds the web-based search interface. This is the longest step — please be patient (takes around 3 to 5 minutes).

> When you run the cell below, you may see some pip dependency conflicts and deprecation warnings. They are safe to ignore.


In [ ]:
#@title Install WISE (run once) { display-mode: "form" }
%cd /content

# 1. Download the latest WISE source code (main branch)
!rm -rf wise
!git clone --quiet --depth 1 -b main https://gitlab.com/vgg/wise/wise.git

# 2. Install WISE and its Python dependencies
#    (uses the CPU build of FAISS, which is robust and plenty fast for small demos)
!pip install -q -r wise/requirements.txt
!pip install -q wise/

# 3. Build the web-based search interface (React + Ant Design)
!cd wise/frontend && npm install --silent && npm run build

%cd /content/wise

# This notebook is set to use a GPU automatically. Check that one is available.
import torch
if torch.cuda.is_available():
    print(f"\nWISE installed. GPU is available ({torch.cuda.get_device_name(0)}).")
else:
    print("\nWISE installed, but NO GPU was detected. The notebook will still work, "
          "but will be much slower.\nTo enable a GPU, go to "
          "Runtime > Change runtime type > Hardware accelerator > GPU, then re-run this cell.")


## Step 2 — Select images

You have two options:

- **Use a demo dataset** — select one from the dropdown (for this exercise, 40 book illustrations from 15th-century Venetian printed books).
- **Upload a ZIP file of images** — select "Upload a ZIP file of images" and choose a single `.zip` archive; all images inside it (including those in sub-folders) will be used.

Uploaded images are prepared automatically for search: any image larger than 800 pixels is resized (keeping its aspect ratio) to speed up feature extraction, and all images are converted to JPEG. If you provide more than 1000 images you will see a warning, since extracting features from that many images can take a long time.


In [ ]:
#@title Choose a demo dataset or upload your own images { display-mode: "form" }
SOURCE = "15th century printed illustration from Venice (40 images)" #@param ["15th century printed illustration from Venice (40 images)", "Vase Images from Oxford Beazley Archive of Greek pottery (804 images)", "Upload a ZIP file of images"]

import os, io, shutil, zipfile
from pathlib import Path
from PIL import Image

# To add more demo datasets later, add an entry here and to the dropdown above.
DATASETS = {
    "15th century printed illustration from Venice (40 images)": {
        "url": "http://thor.robots.ox.ac.uk/wise/assets/test/15ci_selected_40.zip",
        "folder": "15ci_selected_40",
    },
    "Vase Images from Oxford Beazley Archive of Greek pottery (804 images)": {
        "url": "http://thor.robots.ox.ac.uk/wise/assets/test/vases_beazley_archive_selected_804.zip",
        "folder": "vases_beazley_archive_selected_804",
    },
}

MAX_SIZE = 800  # uploaded images larger than this (width or height) are resized

def prepare_uploaded_image(name, data, out_dir):
    """Resize large images and convert to JPEG. Returns True if saved."""
    try:
        img = Image.open(io.BytesIO(data))
    except Exception:
        print(f"  skipped (not a readable image): {name}")
        return False
    img = img.convert("RGB")
    if max(img.size) > MAX_SIZE:
        img.thumbnail((MAX_SIZE, MAX_SIZE))  # resizes in place, keeping aspect ratio
    img.save(os.path.join(out_dir, Path(name).stem + ".jpg"), "JPEG", quality=90)
    return True

if SOURCE in DATASETS:
    # ---- Option A: download a demo dataset ----
    info = DATASETS[SOURCE]
    URL               = info["url"]
    MEDIA_DIRECTORY   = f"/content/wise/wise-data/{info['folder']}"
    PROJECT_DIRECTORY = f"/content/wise/wise-projects/{info['folder']}"

    os.makedirs("/content/wise/wise-data", exist_ok=True)
    !wget -q "{URL}" -O /content/wise/wise-data/dataset.zip
    !unzip -q -o /content/wise/wise-data/dataset.zip -d /content/wise/wise-data/

    # count images recursively, since some datasets are organised in sub-folders
    n_images = sum(1 for p in Path(MEDIA_DIRECTORY).rglob("*")
                   if p.is_file() and not p.name.startswith("."))
    print(f"Downloaded '{SOURCE}' — {n_images} images in {MEDIA_DIRECTORY}")
else:
    # ---- Option B: upload a ZIP file of your own images ----
    from google.colab import files

    MEDIA_DIRECTORY   = "/content/wise/wise-data/my-images"
    PROJECT_DIRECTORY = "/content/wise/wise-projects/my-images"

    # start from a clean folder so re-running does not mix old and new uploads
    if os.path.isdir(MEDIA_DIRECTORY):
        shutil.rmtree(MEDIA_DIRECTORY)
    os.makedirs(MEDIA_DIRECTORY, exist_ok=True)

    # collect a list of (name, image_bytes) pairs from the uploaded ZIP file(s)
    items = []
    print("Select a single ZIP file to upload from your computer...")
    uploaded = files.upload()
    for fname, fdata in uploaded.items():
        if not fname.lower().endswith(".zip"):
            print(f"  skipped (not a .zip file): {fname}")
            continue
        with zipfile.ZipFile(io.BytesIO(fdata)) as zf:
            for member in zf.namelist():
                # skip directories and hidden/metadata files (e.g. __MACOSX)
                if member.endswith("/") or os.path.basename(member).startswith("."):
                    continue
                items.append((member, zf.read(member)))

    if len(items) > 1000:
        print(f"\nWarning: you provided {len(items)} images. Extracting features "
              "from this many images may take a long time. Consider using fewer than 1000.")

    print(f"\nPreparing {len(items)} file(s) (resizing large images, converting to JPEG)...")
    n_images = 0
    for name, data in items:
        if prepare_uploaded_image(name, data, MEDIA_DIRECTORY):
            n_images += 1
    print(f"Prepared {n_images} images in {MEDIA_DIRECTORY}")


## Step 3 — Extract image features

WISE passes every image through a vision-language model (`ViT-B-16-SigLIP2-512`). This turns each image into a numeric embedding (or vector) that captures its visual content, so we can later match images against text descriptions.

> The model is downloaded automatically the first time it is used — this may take a couple of minutes.


In [ ]:
#@title Extract features { display-mode: "form" }
VISION_MODEL = "mlfoundations/open_clip/ViT-B-16-SigLIP2-512/webli"

!python3 extract-features.py "{MEDIA_DIRECTORY}" \
    --project-dir "{PROJECT_DIRECTORY}" \
    --image-feature-id "{VISION_MODEL}" \
    -y


## Step 4 — Build the search index

We build a [FAISS](https://github.com/facebookresearch/faiss) index over the embeddings. This is what makes searching fast and accurate.


In [ ]:
#@title Create the search index { display-mode: "form" }
!python3 create-index.py \
    --project-dir "{PROJECT_DIRECTORY}" \
    --index-type IndexFlatIP


## Step 5 — Search

This starts the WISE web server and opens the search interface right here in the notebook.

Starting the server takes around 30 seconds. Please wait until you see the message **"WISE is running"** and the search interface appears below.

For the **15th century printed illustrations from Venice**, try searches like:
- a boy and a girl talking to each other
- camel
- aesop
- death
- congregation of people
- animals

For the **vase images from the Oxford Beazley Archive of Greek pottery**, try searches like:
- a warrior holding a shield and spear
- a horse
- two figures facing each other
- a winged figure
- a person playing a musical instrument

You can also use the image/upload icon in the interface to search by visual similarity with an example image.

> When you are finished, stop this cell (the stop button) to shut the server down.


In [ ]:
#@title Start the WISE search interface { display-mode: "form" }
import subprocess, time, socket
from pathlib import Path
from google.colab.output import serve_kernel_port_as_iframe

PORT = 9670

# serve.py runs a blocking web server, so we launch it in the background.
server = subprocess.Popen(["python3", "serve.py", "--project-dir", PROJECT_DIRECTORY])

# Wait until the server is ready to accept connections (includes model warm-up).
def wait_for_port(port, timeout=240):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(2)
    return False

if wait_for_port(PORT):
    print("WISE is running. The search interface will appear below.")
    serve_kernel_port_as_iframe(PORT, path=f"/{Path(PROJECT_DIRECTORY).name}/")
else:
    print("The server did not start in time. Re-run this cell, or check the log output above.")


<br><br>

---

## Version history

- v1.0 (1 June 2026) : Initial release for the [ScholarThon](https://blog.humanities.org.uk/2026/05/14/scholarthon-rethinking-the-hackathon-for-arts-and-humanities-research/) event.
